# Gunshot Similarity Search on Lab WAVs

This notebook:
- Recursively loads long `.WAV` files from the lab server folder tree
- Builds a **reference gunshot embedding bank** from an open dataset (UrbanSound8K `gun_shot`)
- Scans each long WAV with **YAMNet embeddings** and finds the most gunshot-like timestamps
- Saves a **spectrogram image** + provides an **audio snippet** around each detected timestamp

> Update the `LAB_ROOT` and `GUNSHOT_ROOT` paths below to match your machine/server.

In [1]:

# --- 1) Paths you should edit ---
from pathlib import Path

# Your lab server folder (matches the screenshot structure)
# Example from screenshot:
# /Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/2-10-25 to 02-24-25
LAB_ROOT = Path("/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/2-10-25 to 02-24-25")

# Where you downloaded UrbanSound8K (gun_shot examples)
# UrbanSound8K typically has audio in: UrbanSound8K/audio/fold*/xxx.wav
GUNSHOT_ROOT = Path("/path/to/UrbanSound8K")

# Limit how many lab files you scan (start small!)
MAX_LAB_FILES = 10

LAB_ROOT, GUNSHOT_ROOT


(PosixPath('/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/2-10-25 to 02-24-25'),
 PosixPath('/path/to/UrbanSound8K'))

In [2]:

# --- 2) Collect lab WAV paths (recursive) ---
import os

lab_files = sorted([p for p in LAB_ROOT.rglob("*.WAV")] + [p for p in LAB_ROOT.rglob("*.wav")])
print("Found lab wavs:", len(lab_files))
lab_files[:5]


Found lab wavs: 154


[PosixPath('/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/2-10-25 to 02-24-25/20250210/20250210_180000.WAV'),
 PosixPath('/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/2-10-25 to 02-24-25/20250210/20250210_190000.WAV'),
 PosixPath('/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/2-10-25 to 02-24-25/20250210/20250210_200000.WAV'),
 PosixPath('/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/2-10-25 to 02-24-25/20250210/20250210_210000.WAV'),
 PosixPath('/Volumes/aid_elephants_interaction/Audio Data/2025/01_FR46/2-10-25 to 02-24-25/20250210/20250210_220000.WAV')]

In [ ]:

# --- 3) Install + load YAMNet ---
%pip -q install tensorflow tensorflow-hub librosa soundfile matplotlib numpy pandas tqdm

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm

import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
TARGET_SR = 16000

print("YAMNet loaded.")


Note: you may need to restart the kernel to use updated packages.


In [ ]:

# --- 4) Helper functions ---
def load_wav_mono_16k(path, sr=TARGET_SR):
    y, _sr = librosa.load(path=str(path), sr=sr, mono=True)
    return y.astype(np.float32), sr

def yamnet_scores_embeddings(y_16k):
    # outputs: (scores, embeddings, spectrogram)
    scores, embeddings, spectrogram = yamnet(y_16k)
    return scores.numpy(), embeddings.numpy(), spectrogram.numpy()

def l2_normalize(x, axis=-1, eps=1e-9):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

def find_urbansound_gunshots(urbansound_root: Path):
    # Common UrbanSound8K layout: UrbanSound8K/audio/fold1/*.wav ... fold10/*.wav
    audio_root = urbansound_root / "audio"
    if not audio_root.exists():
        raise FileNotFoundError(f"Couldn't find {audio_root}. Set GUNSHOT_ROOT correctly.")

    # UrbanSound8K filenames include the class name only in metadata, not filename.
    # Easiest approach: use UrbanSound8K metadata CSV.
    meta_csv = urbansound_root / "metadata" / "UrbanSound8K.csv"
    if not meta_csv.exists():
        raise FileNotFoundError(f"Couldn't find {meta_csv}. UrbanSound8K metadata missing?")

    df = pd.read_csv(meta_csv)
    gun_df = df[df["class"] == "gun_shot"].copy()
    paths = []
    for r in gun_df.itertuples(index=False):
        paths.append(audio_root / f"fold{r.fold}" / r.slice_file_name)
    paths = [p for p in paths if p.exists()]
    return paths

def build_reference_bank(gunshot_paths, max_files=200):
    refs = gunshot_paths[:max_files]
    ref_embs = []
    for p in tqdm(refs, desc="Building reference embeddings"):
        y, _ = load_wav_mono_16k(p)
        _, embs, _ = yamnet_scores_embeddings(y)
        # average over time frames -> one embedding per file
        ref_embs.append(embs.mean(axis=0))
    ref_embs = np.vstack(ref_embs)
    ref_embs = l2_normalize(ref_embs, axis=1)
    return ref_embs, refs

def pick_top_times(scores, times, top_k=5, min_sep_sec=1.0):
    # greedy peak-picking on a 1D score curve
    idx_sorted = np.argsort(scores)[::-1]
    chosen = []
    for idx in idx_sorted:
        t = float(times[idx])
        if all(abs(t - c) >= min_sep_sec for c in chosen):
            chosen.append(t)
        if len(chosen) >= top_k:
            break
    return chosen

def scan_file_for_gunshot_like(wav_path: Path, ref_bank_normed, top_k=5, min_sep_sec=1.0):
    y, sr = load_wav_mono_16k(wav_path)
    _, embs, _ = yamnet_scores_embeddings(y)
    embs_n = l2_normalize(embs, axis=1)

    # similarity per YAMNet frame: max cosine similarity vs any reference
    sim = np.max(embs_n @ ref_bank_normed.T, axis=1)

    # YAMNet has ~0.48s frame hop (approx). We'll use 0.48 for timestamps.
    hop_sec = 0.48
    times = np.arange(len(sim)) * hop_sec

    top_times = pick_top_times(sim, times, top_k=top_k, min_sep_sec=min_sep_sec)

    rows = []
    for t in top_times:
        i = int(round(t / hop_sec))
        rows.append({"file": str(wav_path), "time_sec": t, "similarity": float(sim[i])})

    return pd.DataFrame(rows), sim, times

def save_segment_spectrogram(wav_path: Path, center_time_sec: float, out_png: Path, segment_sec=3.0):
    y, sr = load_wav_mono_16k(wav_path)
    start = max(0, int((center_time_sec - segment_sec/2) * sr))
    end = min(len(y), int((center_time_sec + segment_sec/2) * sr))
    seg = y[start:end]

    S = librosa.stft(seg, n_fft=1024, hop_length=256)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=sr, hop_length=256, x_axis="time", y_axis="hz")
    plt.colorbar(format="%+2.0f dB")
    plt.title(f"{wav_path.name} around t={center_time_sec:.2f}s")
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=200)
    plt.close()
    return out_png

print("Helpers ready.")


In [ ]:

# --- 5) Build gunshot reference embedding bank from UrbanSound8K ---
gunshot_paths = find_urbansound_gunshots(GUNSHOT_ROOT)
print("Gunshot reference clips found:", len(gunshot_paths))
gunshot_paths[:3]


In [ ]:

ref_bank, used_refs = build_reference_bank(gunshot_paths, max_files=200)
ref_bank.shape


In [ ]:

# --- 6) Scan a handful of lab files and rank by best similarity ---
scan_files = lab_files[:MAX_LAB_FILES]
all_hits = []

for p in scan_files:
    hits_df, sim_curve, times = scan_file_for_gunshot_like(p, ref_bank, top_k=3, min_sep_sec=2.0)
    if len(hits_df):
        # Keep best hit per file for ranking
        best = hits_df.sort_values("similarity", ascending=False).iloc[0].to_dict()
        all_hits.append(best)

ranked = pd.DataFrame(all_hits).sort_values("similarity", ascending=False)
ranked


In [ ]:

# --- 7) For the top-ranked file: save spectrograms + show audio snippets ---
from IPython.display import Audio, display

if len(ranked) == 0:
    print("No hits found (or paths wrong). Try increasing MAX_LAB_FILES or verify paths.")
else:
    top_file = Path(ranked.iloc[0]["file"])
    print("Top file:", top_file)

    # rescan to get top 5 times for this file
    hits_df, _, _ = scan_file_for_gunshot_like(top_file, ref_bank, top_k=5, min_sep_sec=2.0)
    display(hits_df)

    for r in hits_df.itertuples(index=False):
        out_png = Path("gunshot_results") / f"{Path(r.file).stem}_t{r.time_sec:.2f}.png"
        save_segment_spectrogram(Path(r.file), r.time_sec, out_png, segment_sec=3.0)
        print("Saved:", out_png)

        y, sr = load_wav_mono_16k(Path(r.file))
        start = max(0, int((r.time_sec - 1.5) * sr))
        end = min(len(y), int((r.time_sec + 1.5) * sr))
        display(Audio(y[start:end], rate=sr))


## Next steps
- Increase `MAX_LAB_FILES` or scan all folders once it works.
- If you want **strict 'gunshot' probability** (not just similarity), we can also read YAMNet class scores and peak-pick on the gunshot-related class.
- If false positives are common, we can train a small classifier on top of the embeddings (gunshot vs non-gunshot) using UrbanSound8K.